# Supply Chain Risk Intelligence Engine

This notebook is a **standalone backend demo**. It does not import from the FastAPI `app/` folder.

It contains the full backend logic inline: data loading, chunking, BM25 retrieval, vector-style retrieval, hybrid retrieval, reranking, query rewriting, multi-query retrieval, multi-hop retrieval, answer generation, and evaluation.

## Real Problem Statement

Supply chain teams need to answer questions that connect suppliers, shipments, components, BOM mappings, products, compliance reports, incident reports, and regional risk memos.

Example:

> Which products are most exposed to delayed semiconductor shipments from Malaysia?

This requires multiple hops: supplier/country detection, shipment and incident evidence, component-to-product mapping through BOM rows, alternate supplier coverage, and a cited business-risk answer.

## 1. Setup

OpenAI is optional. Set `USE_OPENAI_LLM=true` and `OPENAI_API_KEY=...` in `.env` if you want GPT-based answer synthesis. The retrieval code remains local and visible for demo/explanation.

In [36]:
from __future__ import annotations

import json
import math
import os
import hashlib
import re
from collections import Counter
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi

load_dotenv()

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "company_data"

CHUNK_SIZE = 900       # maximum characters in each retrievable passage
CHUNK_OVERLAP = 140    # context repeated between neighboring chunks
TOP_K = 8              # final evidence chunks returned to the user
CANDIDATE_K = 30       # broad candidate pool before reranking

USE_OPENAI_LLM = os.getenv("USE_OPENAI_LLM", "false").lower() == "true"
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

DATA_DIR

WindowsPath('d:/Users data/Documents/Project_7/data/company_data')

## 2. Data Check

If this folder is missing, run this from the project root:

```bash
python -m data.generate_data
```

In [37]:
manifest_path = DATA_DIR / "manifest.json"
if not manifest_path.exists():
    raise FileNotFoundError("Run `python -m data.generate_data` from project root first.")

manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
manifest

{'suppliers': 150,
 'components': 400,
 'products': 80,
 'bom_rows': 562,
 'shipments': 2500,
 'purchase_orders': 1500,
 'scorecard_rows': 1200,
 'documents': 540,
 'eval_questions': 60}

## 3. Load Markdown Documents And CSV Tables

The backend uses unstructured reports and structured enterprise tables. CSV rows are converted into searchable documents so shipments, BOM rows, suppliers, and products can be retrieved alongside Markdown reports.

In [38]:
def load_markdown_documents(data_dir: Path) -> list[Document]:
    """Read generated Markdown files and attach source metadata."""
    documents = []
    for path in sorted((data_dir / "documents").rglob("*.md")):
        documents.append(
            Document(
                page_content=path.read_text(encoding="utf-8"),
                metadata={
                    "source": path.relative_to(data_dir).as_posix(),
                    "document_type": path.parent.name,
                    "file_name": path.name,
                },
            )
        )
    return documents


def load_tables(data_dir: Path) -> dict[str, pd.DataFrame]:
    """Load structured CSV datasets into pandas DataFrames."""
    return {path.stem: pd.read_csv(path) for path in sorted((data_dir / "tables").glob("*.csv"))}


def table_rows_as_documents(tables: dict[str, pd.DataFrame]) -> list[Document]:
    """Convert each row into a searchable text document with table metadata."""
    documents = []
    for table_name, frame in tables.items():
        for row_index, row in frame.iterrows():
            row_text = "\n".join(f"{key}: {value}" for key, value in row.dropna().to_dict().items())
            documents.append(
                Document(
                    page_content=f"Table: {table_name}\n{row_text}",
                    metadata={
                        "source": f"tables/{table_name}.csv#{row_index}",
                        "document_type": "structured_row",
                        "table": table_name,
                        "row_index": int(row_index),
                    },
                )
            )
    return documents


tables = load_tables(DATA_DIR)
raw_documents = load_markdown_documents(DATA_DIR) + table_rows_as_documents(tables)

print("Tables:", list(tables.keys()))
print("Document objects:", len(raw_documents))

Tables: ['bill_of_materials', 'components', 'products', 'purchase_orders', 'shipments', 'supplier_scorecards', 'suppliers']
Document objects: 6932


## 4. Chunk Documents

Chunking turns long reports and table rows into retrievable passages. Overlap prevents losing context when a supplier name and its risk explanation fall near a boundary.

In [49]:
def chunk_documents(documents: list[Document], chunk_size: int, chunk_overlap: int) -> list[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(documents)
    for index, chunk in enumerate(chunks):
        chunk.metadata["chunk_id"] = f"CHUNK-{index:06d}"
    return chunks


chunks = chunk_documents(raw_documents, CHUNK_SIZE, CHUNK_OVERLAP)
len(chunks), chunks[1562].metadata

(7641,
 {'source': 'tables/bill_of_materials.csv#313',
  'document_type': 'structured_row',
  'table': 'bill_of_materials',
  'row_index': 313,
  'chunk_id': 'CHUNK-001562'})

## 5. Build Search Indexes

> **Backend idea:** this project uses two different retrieval signals because supply chain questions need both exact matching and semantic understanding.

**BM25 keyword index**

BM25 is useful when the question contains exact operational terms such as supplier names, ports, country names, shipment IDs, component IDs, or product names. If the user asks about `Malaysia`, `SUP-0009`, or `Port Klang`, lexical search should strongly reward chunks that contain those exact terms.

**OpenAI embedding index**

Embeddings are useful when the user asks a vague business question such as "which suppliers are risky" or "what products are exposed". The documents may not use the same words, but embeddings can still retrieve semantically related evidence about delays, single-source exposure, compliance findings, and product impact.

> **Why this is industry-style:** production RAG systems rarely depend on only one search method. Exact search protects entity precision, while embeddings improve semantic recall.

**Caching**

The OpenAI embedding matrix is cached locally under `data/company_data/cache/`. This avoids paying for and recomputing document embeddings every time the notebook runs. Query embeddings are still computed live because every user question is different.

In [40]:
TOKEN_PATTERN = re.compile(r"[A-Za-z0-9_\-]+")


def tokenize(text: str) -> list[str]:
    """Split text into normalized tokens for BM25 and reranking.

    Args:
        text: Raw document, chunk, or query text.

    Returns:
        Lowercase tokens. Hyphenated operational IDs such as SUP-0001 remain searchable.
    """
    return [token.lower() for token in TOKEN_PATTERN.findall(text)]


def embedding_cache_path(chunks: list[Document], model_name: str) -> Path:
    """Create a stable cache path for the current chunks and embedding model.

    Args:
        chunks: Final retrieval chunks. Their content is hashed so stale caches are avoided.
        model_name: OpenAI embedding model name, for example `text-embedding-3-small`.

    Returns:
        Path to a JSON file containing cached embeddings for this exact dataset state.
    """
    digest = hashlib.sha256()
    digest.update(model_name.encode("utf-8"))
    for chunk in chunks:
        digest.update(str(chunk.metadata.get("chunk_id", "")).encode("utf-8"))
        digest.update(chunk.page_content.encode("utf-8"))
    cache_dir = DATA_DIR / "cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    return cache_dir / f"openai_embeddings_{digest.hexdigest()[:16]}.json"


def load_or_create_openai_embeddings(chunks: list[Document], model_name: str) -> tuple[OpenAIEmbeddings, np.ndarray]:
    """Load cached chunk embeddings or create them with OpenAI.

    Args:
        chunks: Retrieval chunks that should be embedded.
        model_name: OpenAI embedding model. `text-embedding-3-small` is a strong default for demos.

    Returns:
        A tuple of `(embedding_client, embedding_matrix)` where the matrix has one row per chunk.

    Raises:
        RuntimeError: If `OPENAI_API_KEY` is not set. This notebook requires OpenAI embeddings.
    """
    if not OPENAI_API_KEY:
        raise RuntimeError("OPENAI_API_KEY is required because this notebook uses OpenAI embeddings for vector search.")
    client = OpenAIEmbeddings(model=model_name, api_key=OPENAI_API_KEY)
    cache_path = embedding_cache_path(chunks, model_name)
    if cache_path.exists():
        payload = json.loads(cache_path.read_text(encoding="utf-8"))
        return client, np.asarray(payload["embeddings"], dtype=np.float32)
    embeddings = client.embed_documents([chunk.page_content for chunk in chunks])
    cache_path.write_text(json.dumps({"model": model_name, "embeddings": embeddings}), encoding="utf-8")
    return client, np.asarray(embeddings, dtype=np.float32)


def cosine_scores(query_vector: list[float], embedding_matrix: np.ndarray) -> np.ndarray:
    """Calculate cosine similarity between a query embedding and all chunk embeddings.

    Args:
        query_vector: OpenAI embedding vector for the user query.
        embedding_matrix: Matrix of OpenAI embeddings for all chunks.

    Returns:
        One similarity score per chunk. Higher values mean stronger semantic match.
    """
    query = np.asarray(query_vector, dtype=np.float32)
    query_norm = np.linalg.norm(query)
    matrix_norms = np.linalg.norm(embedding_matrix, axis=1)
    return embedding_matrix.dot(query) / np.maximum(matrix_norms * query_norm, 1e-12)


texts = [chunk.page_content for chunk in chunks]
tokenized_texts = [tokenize(text) for text in texts]

bm25_index = BM25Okapi(tokenized_texts)
embedding_client, embedding_matrix = load_or_create_openai_embeddings(chunks, OPENAI_EMBEDDING_MODEL)

embedding_matrix.shape

(7641, 1536)

## 6. Retrieval Functions

> **What this section builds:** reusable backend retrieval functions. These are the core primitives behind the notebook demo and the FastAPI backend.

**Vector search**

The query is embedded using OpenAI, compared against the cached document embedding matrix, and ranked by cosine similarity. This helps retrieve conceptually related evidence even when the words differ.

**BM25 search**

The query is tokenized and scored against the BM25 index. This is strong for exact supplier names, product IDs, countries, ports, and compliance phrases.

**Hybrid search**

Hybrid retrieval runs both vector search and BM25, then combines the ranked lists using Reciprocal Rank Fusion. This gives us candidates that are both semantically relevant and lexically precise.

**Reranking**

The reranker takes a broader candidate pool and reorders it against the original user question. Retrieval is about finding possible evidence; reranking is about choosing the best evidence for the final answer.

In [41]:
def vector_search(query: str, top_k: int = TOP_K) -> list[tuple[Document, float]]:
    """Search chunks by semantic similarity using OpenAI embeddings.

    Args:
        query: User question or rewritten retrieval query.
        top_k: Number of highest-scoring chunks to return.

    Returns:
        Ranked `(Document, score)` pairs where score is cosine similarity.
    """
    query_vector = embedding_client.embed_query(query)
    scores = cosine_scores(query_vector, embedding_matrix)
    indexes = np.argsort(scores)[::-1][:top_k]
    return [(chunks[int(index)], float(scores[int(index)])) for index in indexes if scores[int(index)] > 0]


def bm25_search(query: str, top_k: int = TOP_K) -> list[tuple[Document, float]]:
    """Search by BM25 keyword relevance."""
    scores = bm25_index.get_scores(tokenize(query))
    indexes = np.argsort(scores)[::-1][:top_k]
    return [(chunks[int(index)], float(scores[int(index)])) for index in indexes if scores[int(index)] > 0]


def reciprocal_rank_fusion(result_sets: list[list[tuple[Document, float]]], rrf_k: int = 60) -> list[tuple[Document, float]]:
    """Fuse multiple ranked lists and reward chunks that rank well repeatedly."""
    fused = {}
    for results in result_sets:
        for rank, (doc, _score) in enumerate(results, start=1):
            key = doc.metadata.get("chunk_id") or doc.metadata.get("source") or doc.page_content[:80]
            contribution = 1.0 / (rrf_k + rank)
            fused[key] = (doc, fused.get(key, (doc, 0.0))[1] + contribution)
    return sorted(fused.values(), key=lambda item: item[1], reverse=True)


def hybrid_search(query: str, top_k: int = TOP_K, candidate_k: int = CANDIDATE_K) -> list[tuple[Document, float]]:
    """Combine BM25 and vector-style retrieval."""
    return reciprocal_rank_fusion([
        vector_search(query, top_k=candidate_k),
        bm25_search(query, top_k=candidate_k),
    ])[:top_k]


def rerank(query: str, candidates: list[tuple[Document, float]], top_k: int = TOP_K) -> list[tuple[Document, float]]:
    """Rerank candidates using overlap, evidence type, exactness, and specificity."""
    query_terms = set(tokenize(query))
    reranked = []
    for doc, base_score in candidates:
        doc_terms = tokenize(doc.page_content)
        counts = Counter(doc_terms)
        overlap = sum(counts[term] for term in query_terms)
        exact_bonus = 0.2 if query.lower() in doc.page_content.lower() else 0.0
        evidence_bonus = 0.08 if doc.metadata.get("document_type") in {"incident_reports", "risk_reports", "compliance_reports", "structured_row"} else 0.0
        length_penalty = 1.0 / math.sqrt(max(1, len(doc_terms) / 120))
        score = float(base_score) + overlap * 0.035 * length_penalty + exact_bonus + evidence_bonus
        reranked.append((doc, score))
    return sorted(reranked, key=lambda item: item[1], reverse=True)[:top_k]


## 7. Query Rewrite, Multi-Query, And Decomposition

> **Backend idea:** analyst questions are often underspecified. Query transformation improves retrieval before search even starts.

**Query rewrite**

A user might ask: "Which products are exposed to Malaysia delays?" The system rewrites this into a richer retrieval query containing terms like supplier, shipment, component, BOM, product impact, delay evidence, single-source exposure, and logistics risk.

**Multi-query retrieval**

Instead of trusting one query, the system creates multiple search variants. One variant may target shipment delays, another may target incident reports, another may target BOM/product impact, and another may target compliance or alternate suppliers.

**Question decomposition**

Complex questions are broken into smaller retrieval hops. For supply chain risk, this usually means: identify entities, find risk evidence, map suppliers/components to products, and find mitigation or alternate-source evidence.

> **OpenAI usage:** when `USE_OPENAI_LLM=true`, rewrite, multi-query generation, and decomposition are done with `ChatOpenAI`. The deterministic fallback exists only so the notebook remains explainable if LLM calls are disabled.

In [42]:
COUNTRIES = ["Malaysia", "Vietnam", "China", "Taiwan", "South Korea", "India", "Mexico", "Poland", "Germany", "Turkey"]
RISK_WORDS = ["delay", "risk", "compliance", "single-source", "shortage", "congestion", "quality", "capacity", "export"]


def extract_entities(question: str) -> dict[str, list[str]]:
    countries = [country for country in COUNTRIES if country.lower() in question.lower()]
    ids = re.findall(r"\b(?:SUP|CMP|PRD|SHP|PO)-\d+\b", question, flags=re.IGNORECASE)
    phrases = re.findall(r"\b[A-Z][A-Za-z]+(?:\s[A-Z][A-Za-z]+){0,3}\b", question)
    return {"countries": countries, "ids": [item.upper() for item in ids], "phrases": sorted(set(phrases))[:8]}


def rewrite_query(question: str) -> str:
    entities = extract_entities(question)
    intent_terms = [word for word in RISK_WORDS if word in question.lower()]
    additions = ["supplier", "component", "product", "shipment", "BOM", "evidence"]
    if "product" in question.lower():
        additions.extend(["bill of materials", "single source", "revenue impact"])
    if "compliance" in question.lower():
        additions.extend(["audit", "certification", "screening"])
    entity_text = " ".join(entities["countries"] + entities["ids"] + entities["phrases"])
    return f"{question} {entity_text} {' '.join(intent_terms)} {' '.join(additions)}".strip()


def generate_query_variants(question: str) -> list[str]:
    variants = [
        rewrite_query(question),
        f"{question} delayed shipments supplier component product impact",
        f"{question} risk report incident report logistics memo compliance review",
        f"{question} single-source supplier alternate supplier bill of materials",
        f"{question} procurement mitigation operational exposure",
    ]
    unique = []
    seen = set()
    for variant in variants:
        compact = " ".join(variant.split())
        if compact.lower() not in seen:
            unique.append(compact)
            seen.add(compact.lower())
    return unique


def decompose_question(question: str) -> list[str]:
    return [
        f"Identify suppliers, countries, components, and products mentioned in: {question}",
        f"Find shipment delays, incidents, risk reports, and compliance issues related to: {question}",
        f"Map affected suppliers and components to finished products using BOM data for: {question}",
        f"Find alternate supplier coverage and business impact evidence for: {question}",
    ]


## 8. Compare Basic Strategies

> **Demo goal:** run the same business question through several retrieval strategies and compare the evidence quality.

The strategies are intentionally separated so you can explain why each layer exists:

- **Vector only:** good semantic recall, but may miss exact operational entities.
- **BM25 only:** strong exact matching, but can miss semantically relevant documents.
- **Hybrid:** combines semantic and exact matching.
- **Hybrid plus rerank:** retrieves broadly, then promotes the most useful evidence.

This is the heart of the retrieval-quality story. The project is not just answering questions; it is showing why better retrieval architecture produces better evidence.

In [43]:
question = "Which products are most exposed to delayed semiconductor shipments from Malaysia?"
rewritten = rewrite_query(question)

def preview(results: list[tuple[Document, float]], n: int = 5) -> pd.DataFrame:
    return pd.DataFrame([
        {
            "source": doc.metadata.get("source"),
            "type": doc.metadata.get("document_type"),
            "score": round(float(score), 4),
            "snippet": " ".join(doc.page_content.split())[:260],
        }
        for doc, score in results[:n]
    ])

vector_results = vector_search(question, top_k=5)
bm25_results = bm25_search(question, top_k=5)
hybrid_results = hybrid_search(rewritten, top_k=8, candidate_k=30)
reranked_results = rerank(question, hybrid_search(rewritten, top_k=30, candidate_k=50), top_k=8)

preview(reranked_results)

,source,type,score,snippet
0,documents/incident_reports/incident_0061_SUP-0...,incident_reports,0.4628,## Recommended Response - check high-revenue p...
1,documents/incident_reports/incident_0018_SUP-0...,incident_reports,0.4618,## Recommended Response - review active shipme...
2,documents/risk_reports/risk_report_0053_malays...,risk_reports,0.4577,# Risk Report - Malaysia semiconductor Report ...
3,documents/compliance_reports/compliance_0047_S...,compliance_reports,0.4083,# Compliance Review - Shenzhen Materials Malay...
4,documents/incident_reports/incident_0061_SUP-0...,incident_reports,0.3918,# Shipment Incident INC-2026-0061 Incident ID:...


## 9. Multi-Query Retrieval

> **Why multi-query matters:** one query rarely captures every way evidence may be written across enterprise data.

For example, the same risk may appear in different places using different language:

- a shipment table may say `delay_days = 12`
- an incident report may say `actual arrival moved beyond planned arrival`
- a supplier profile may say `recent shipment review found delayed shipments`
- a logistics memo may say `port congestion can cascade into production risk`

Multi-query retrieval creates several query variants, searches each one, fuses the results, and reranks the final evidence. This improves recall without flooding the answer generator with random context.

In [44]:
def multi_query_retrieve(question: str, top_k: int = TOP_K) -> tuple[list[str], list[tuple[Document, float]]]:
    variants = generate_query_variants(question)
    result_sets = [rerank(variant, hybrid_search(variant, top_k=CANDIDATE_K, candidate_k=50), top_k=top_k) for variant in variants]
    fused = reciprocal_rank_fusion(result_sets)
    final_results = rerank(question, fused, top_k=top_k)
    return variants, final_results

variants, multi_query_results = multi_query_retrieve(question)
variants, preview(multi_query_results)

(['Which products are most exposed to delayed semiconductor shipments from Malaysia? Malaysia Malaysia Which delay supplier component product shipment BOM evidence bill of materials single source revenue impact',
  'Which products are most exposed to delayed semiconductor shipments from Malaysia? delayed shipments supplier component product impact',
  'Which products are most exposed to delayed semiconductor shipments from Malaysia? risk report incident report logistics memo compliance review',
  'Which products are most exposed to delayed semiconductor shipments from Malaysia? single-source supplier alternate supplier bill of materials',
  'Which products are most exposed to delayed semiconductor shipments from Malaysia? procurement mitigation operational exposure'],
                                               source              type  \
 0  documents/incident_reports/incident_0015_SUP-0...  incident_reports   
 1  documents/incident_reports/incident_0030_SUP-0...  incident_reports

## 10. Multi-Hop Retrieval

> **Why multi-hop matters:** supply chain risk questions are relationship questions, not single-document questions.

A question like "Which products are exposed to delayed semiconductor shipments from Malaysia?" requires a chain:

1. Find Malaysian semiconductor suppliers.
2. Find shipment delays or incident evidence.
3. Identify affected components.
4. Map components to products through BOM rows.
5. Check single-source and alternate supplier coverage.

The multi-hop function creates sub-questions, retrieves evidence for each hop, fuses all hop results, and reranks them against the original question. This is what makes the project feel like a real risk intelligence system rather than a simple document chatbot.

In [45]:
def multi_hop_retrieve(question: str, top_k: int = TOP_K) -> dict[str, Any]:
    rewritten = rewrite_query(question)
    hops = []
    result_sets = []
    for hop_index, sub_question in enumerate(decompose_question(question), start=1):
        hop_query = f"{sub_question} {rewritten}"
        hop_results = rerank(hop_query, hybrid_search(hop_query, top_k=CANDIDATE_K, candidate_k=50), top_k=top_k)
        hops.append({
            "hop": hop_index,
            "query": hop_query,
            "top_sources": [doc.metadata.get("source") for doc, _score in hop_results[:3]],
        })
        result_sets.append(hop_results)
    fused = reciprocal_rank_fusion(result_sets)
    final_results = rerank(question, fused, top_k=top_k)
    return {"rewritten_query": rewritten, "hops": hops, "results": final_results}

multi_hop = multi_hop_retrieve(question)
multi_hop["hops"]

[{'hop': 1,
  'query': 'Identify suppliers, countries, components, and products mentioned in: Which products are most exposed to delayed semiconductor shipments from Malaysia? Which products are most exposed to delayed semiconductor shipments from Malaysia? Malaysia Malaysia Which delay supplier component product shipment BOM evidence bill of materials single source revenue impact',
  'top_sources': ['documents/compliance_reports/compliance_0047_SUP-0054.md',
   'documents/incident_reports/incident_0061_SUP-0054.md',
   'documents/incident_reports/incident_0018_SUP-0054.md']},
 {'hop': 2,
  'query': 'Find shipment delays, incidents, risk reports, and compliance issues related to: Which products are most exposed to delayed semiconductor shipments from Malaysia? Which products are most exposed to delayed semiconductor shipments from Malaysia? Malaysia Malaysia Which delay supplier component product shipment BOM evidence bill of materials single source revenue impact',
  'top_sources': ['

## 11. Answer Generation

> **What this step does:** convert retrieved evidence into a business-ready answer.

The answer generator receives only the selected evidence chunks. This keeps the model grounded and makes citations possible.

When `USE_OPENAI_LLM=true`, OpenAI synthesizes the final response with:

- direct answer
- key risk drivers
- affected suppliers, components, and products
- recommended actions
- citations using source names

The notebook also includes a deterministic local answer format so the retrieval outputs can still be inspected without relying on the final LLM call.

In [46]:
def evidence_payload(results: list[tuple[Document, float]], max_chars: int = 900) -> list[dict[str, Any]]:
    return [
        {
            "source": doc.metadata.get("source", "unknown"),
            "score": round(float(score), 4),
            "text": doc.page_content[:max_chars],
            "metadata": doc.metadata,
        }
        for doc, score in results
    ]


def risk_score(results: list[tuple[Document, float]]) -> float:
    text = "\n".join(doc.page_content.lower() for doc, _score in results)
    score = 0.25
    for term, weight in {
        "delayed": 0.14,
        "single_source_flag: yes": 0.18,
        "single-source": 0.12,
        "critical": 0.10,
        "compliance": 0.08,
        "capacity": 0.07,
        "congestion": 0.08,
        "no approved alternate": 0.14,
    }.items():
        if term in text:
            score += weight
    return round(min(score, 0.98), 2)


def generate_local_answer(question: str, results: list[tuple[Document, float]], strategy: str) -> str:
    lines = [f"Strategy: {strategy}", f"Risk score: {risk_score(results)}", "", "Key evidence:"]
    for doc, score in results[:5]:
        source = doc.metadata.get("source", "unknown")
        snippet = " ".join(doc.page_content.replace("#", "").split())[:360]
        lines.append(f"- {snippet} [source: {source}, score: {score:.3f}]")
    lines.append("")
    lines.append("Recommended action: validate affected products through BOM rows, prioritize tier_1 or high-revenue products, and confirm alternate supplier readiness before escalating mitigation.")
    return "\n".join(lines)


def generate_openai_answer(question: str, results: list[tuple[Document, float]], strategy: str) -> str:
    from langchain_openai import ChatOpenAI

    evidence_text = "\n\n".join(
        f"SOURCE: {item['source']}\nSCORE: {item['score']}\nTEXT:\n{item['text']}"
        for item in evidence_payload(results, max_chars=1200)
    )
    prompt = f"""
You are a supply chain risk analyst. Answer using only the evidence below.
Include direct answer, risk drivers, affected entities, recommended actions, and citations.

Question: {question}
Strategy: {strategy}

Evidence:
{evidence_text}
"""
    llm = ChatOpenAI(model=OPENAI_MODEL, temperature=0)
    return llm.invoke(prompt).content


def generate_answer(question: str, results: list[tuple[Document, float]], strategy: str) -> str:
    if USE_OPENAI_LLM and OPENAI_API_KEY:
        return generate_openai_answer(question, results, strategy)
    return generate_local_answer(question, results, strategy)

answer = generate_answer(question, multi_hop["results"], strategy="multi_hop")
print(answer)

**Direct Answer:** The products most exposed to delayed semiconductor shipments from Malaysia are the Helios Energy Controller 06, Helios Energy Controller 18, Helios Energy Controller 50, Helios Energy Controller 24, Helios Energy Controller 14, and Helios Energy Controller 46.

**Risk Drivers:**
- Customs inspection delays at Port Klang (moderate severity).
- Single-source product-component mappings for affected components.

**Affected Entities:**
- Helios Energy Controller 06
- Helios Energy Controller 18
- Helios Energy Controller 50
- Helios Energy Controller 24
- Helios Energy Controller 14
- Helios Energy Controller 46
- Other products: Zenith Edge Server 04, Pulse Wearable Dock 42, Pulse Wearable Dock 35, Orion Smart Hub 56, Nova Health Monitor 58, Nova Health Monitor 15, Atlas Robotics Kit 09, Atlas Robotics Kit 39, Atlas Robotics Kit 43.

**Recommended Actions:**
- Review active shipments from Port Klang.
- Validate whether alternate suppliers are production-qualified.
- Chec

## 12. Evaluation

> **Why evaluation matters:** a retrieval project should not only look good in one demo question. It should measure whether better strategies retrieve more correct evidence.

The generated eval set contains business questions plus expected entities. This notebook computes a simple `entity_recall` score by checking whether retrieved evidence contains the expected suppliers, components, products, countries, or categories.

This lets us compare strategies such as:

- vector only
- BM25 only
- hybrid
- hybrid plus rerank
- multi-query
- multi-hop

In a production version, this can be expanded into Recall@K, MRR, NDCG, citation coverage, answer faithfulness, latency, and cost per query.

In [47]:
def entity_recall(retrieved_text: str, expected_entities: list[str]) -> float:
    if not expected_entities:
        return 0.0
    lower = retrieved_text.lower()
    hits = sum(1 for entity in expected_entities if str(entity).lower() in lower)
    return round(hits / len(expected_entities), 3)


def retrieve_by_strategy(question: str, strategy: str, top_k: int = TOP_K) -> list[tuple[Document, float]]:
    if strategy == "vector":
        return vector_search(question, top_k=top_k)
    if strategy == "bm25":
        return bm25_search(question, top_k=top_k)
    if strategy == "hybrid":
        return hybrid_search(rewrite_query(question), top_k=top_k, candidate_k=CANDIDATE_K)
    if strategy == "hybrid_rerank":
        return rerank(question, hybrid_search(rewrite_query(question), top_k=CANDIDATE_K, candidate_k=50), top_k=top_k)
    if strategy == "multi_query":
        return multi_query_retrieve(question, top_k=top_k)[1]
    if strategy == "multi_hop":
        return multi_hop_retrieve(question, top_k=top_k)["results"]
    raise ValueError(strategy)


def run_eval(strategy: str, limit: int = 20) -> dict[str, Any]:
    eval_items = json.loads((DATA_DIR / "eval" / "eval_questions.json").read_text(encoding="utf-8"))[:limit]
    rows = []
    for item in eval_items:
        results = retrieve_by_strategy(item["question"], strategy=strategy, top_k=8)
        retrieved_text = "\n".join(doc.page_content for doc, _score in results)
        rows.append({
            "question_id": item["question_id"],
            "strategy": strategy,
            "entity_recall": entity_recall(retrieved_text, item.get("expected_entities", [])),
            "top_source": results[0][0].metadata.get("source") if results else "none",
        })
    return {
        "strategy": strategy,
        "average_entity_recall": round(sum(row["entity_recall"] for row in rows) / max(1, len(rows)), 3),
        "rows": rows,
    }

reports = [run_eval(strategy, limit=20) for strategy in ["vector", "bm25", "hybrid", "hybrid_rerank", "multi_query", "multi_hop"]]
pd.DataFrame([{"strategy": r["strategy"], "average_entity_recall": r["average_entity_recall"]} for r in reports])

,strategy,average_entity_recall
0,vector,0.870
1,bm25,0.767
2,hybrid,0.830
3,hybrid_rerank,0.860
4,multi_query,0.918
5,multi_hop,0.890


## 13. Final Evidence Objects

In [48]:
pd.DataFrame(evidence_payload(multi_hop["results"]))[["source", "score", "text"]].head(8)

,source,score,text
0,documents/incident_reports/incident_0015_SUP-0...,0.4802,## Business Impact Assessment\n\nAffected fini...
1,documents/risk_reports/risk_report_0053_malays...,0.4764,# Risk Report - Malaysia semiconductor\n\nRepo...
2,documents/incident_reports/incident_0035_SUP-0...,0.4750,## Business Impact Assessment\n\nAffected fini...
3,documents/incident_reports/incident_0073_SUP-0...,0.4459,## Business Impact Assessment\n\nAffected fini...
4,documents/incident_reports/incident_0006_SUP-0...,0.4456,## Business Impact Assessment\n\nAffected fini...
5,documents/incident_reports/incident_0088_SUP-0...,0.4454,## Business Impact Assessment\n\nAffected fini...
6,documents/incident_reports/incident_0061_SUP-0...,0.4452,## Recommended Response\n\n- check high-revenu...
7,documents/incident_reports/incident_0018_SUP-0...,0.4449,## Recommended Response\n\n- review active shi...
